# A flight recorder for one agent run

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 23 §23.1–23.3 · type: walkthrough*

**One-line promise:** instrument an agent loop so a single user request becomes a single OpenTelemetry **trace** — every model call, tool call, and retrieval a span with the right attributes — then compute per-run cost straight from those spans.


## 🧠 Why this matters

Evaluation tells you *how good* the system is; observability tells you *what it actually did*. For deterministic software you can re-run the code and reproduce the bug. An agent denies you that: the run that failed at 3 a.m. may never happen again, because the model sampled differently, retrieval changed, or a tool returned something it will never return again. The **trace of that run is the only evidence that will ever exist**.

A trace is a *flight recorder for one agent run* — a tree of timed **spans** (one operation each, with attributes and parent→child edges meaning "this happened inside that"). Design your instrumentation by asking one question: *if this run misbehaves once and never again, does the trace alone let me explain it?* If not, you are missing a span or an attribute.


## Objectives & prerequisites

**By the end you can:**
- Stand up an OpenTelemetry `TracerProvider` with an **in-memory exporter** — no backend, no network — so spans are inspectable right in the notebook.
- Instrument the book's three spans — `agent.run`, `llm.call`, `tool.call` — using the OTel **GenAI semantic conventions** (`gen_ai.request.model`, `gen_ai.usage.input_tokens`, …).
- Read a **span tree** to see what a run actually did, including a *failed tool call and its retry* a final-answer-only log would hide.
- Record exceptions on the failing span, compute **per-run cost** from span attributes, and propagate trace context across a sub-agent boundary.

**Prereqs:** Ch 12 (the tool-use loop being instrumented); §21.1's rule that *every run must emit a trace*. Mock mode (default) needs no API key.

**Extra package beyond `requirements.txt`:** none — `opentelemetry-sdk` is already pinned. (Mock agent + tools are pure stdlib; no Anthropic call is made in `MOCK=1`.)


## Setup

We default to `MOCK=1`: the traced agent returns **canned, realistic** responses so this notebook runs free, offline, and deterministically with no API key. Set `COMPANION_MOCK=0` (and `ANTHROPIC_API_KEY`) only if you want to trace a live model call — the tracing layer itself is identical either way.


In [ ]:
import json
import os
import random

from dotenv import load_dotenv

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import (
    InMemorySpanExporter,
)
from opentelemetry.trace import StatusCode

load_dotenv()
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
SEED = 23
random.seed(SEED)

# In MOCK=0 you would read the key here; we never hardcode it.
if not MOCK and not os.getenv("ANTHROPIC_API_KEY"):
    raise SystemExit(
        "MOCK=0 but ANTHROPIC_API_KEY is not set. "
        "Set it in your .env, or unset COMPANION_MOCK to run offline."
    )

print(f"mode = {'MOCK (offline, free)' if MOCK else 'LIVE (real API calls)'}  | seed = {SEED}")
print("otel exporter = in-memory (no collector, no network)")

### Wire up an in-memory tracer

A `TracerProvider` is the root of OTel emission; a **span processor** forwards finished spans to an **exporter**. In production the exporter ships to Langfuse / LangSmith / an OTLP collector. Here we use `InMemorySpanExporter`, which simply keeps finished spans in a Python list — the whole point of the chapter's *swappable backend* advice: the instrumentation never changes, only this one line does.


In [ ]:
exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("capstone.agent")

def reset_spans():
    """Clear captured spans so a cell can be re-run cleanly."""
    exporter.clear()

print("tracer ready:", tracer)

### The mock model and tools (what we're instrumenting)

These stand in for the Ch 12 tool-use loop. `call_model` returns a canned response object shaped like the Anthropic SDK's (`.model`, `.usage.input_tokens`, `.usage.output_tokens`, `.stop_reason`) so the instrumentation reads exactly the same attributes it would from a live response. One tool, `send_report`, is rigged to **fail on its first call and succeed on retry** — the kind of transient failure a final-answer-only log erases.


In [ ]:
from dataclasses import dataclass

MODEL = "claude-sonnet-4-6"

@dataclass
class Usage:
    input_tokens: int
    output_tokens: int

@dataclass
class Resp:
    model: str
    usage: Usage
    stop_reason: str
    text: str

# Canned, deterministic model turns (input_tokens, output_tokens, stop_reason, text).
_CANNED_TURNS = [
    (1850, 90, "tool_use", "I'll look up the account, then send the summary report."),
    (2120, 140, "tool_use", "Retrying the report send."),
    (2310, 210, "end_turn", "Done — account is in good standing; report sent."),
]

def call_model(messages, turn_idx, **params):
    """Mock of capstone/telemetry.py call_model: returns an SDK-shaped response."""
    if MOCK:
        t_in, t_out, stop, text = _CANNED_TURNS[turn_idx]
        return Resp(MODEL, Usage(t_in, t_out), stop, text)
    # --- LIVE path (only runs when MOCK=0; costs ~a few hundred tokens/turn) ---
    from anthropic import Anthropic
    client = Anthropic()
    r = client.messages.create(model=MODEL, max_tokens=512, messages=messages, **params)
    return Resp(r.model, Usage(r.usage.input_tokens, r.usage.output_tokens),
                r.stop_reason, r.content[0].text if r.content else "")

_send_report_attempts = {"n": 0}

def tool_lookup_account(account_id):
    return {"account_id": account_id, "status": "good_standing", "balance": 0.0}

def tool_send_report(to, body):
    _send_report_attempts["n"] += 1
    if _send_report_attempts["n"] == 1:
        raise ConnectionError("report service 503: upstream timeout")
    return {"delivered": True, "to": to, "bytes": len(body)}

TOOLS = {"lookup_account": tool_lookup_account, "send_report": tool_send_report}
print("mock model + tools ready (send_report fails once, then succeeds)")

### Instrument the three spans (the book's `telemetry.py` shapes)

These three context managers are the heart of the chapter. Note the attribute choices:
- `agent.run` (root): `agent.name`, `user.id` (**an id, never PII**), `session.id`, and — set at the end — `agent.steps` and `agent.outcome` (`ok|error|gaveup`).
- `llm.call`: the OTel **GenAI** names — `gen_ai.request.model`, `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens`, `gen_ai.response.finish_reason`.
- `tool.call`: tool name, **truncated** args, result size — and on failure, `record_exception` + `StatusCode.ERROR` so the error is captured *in the tree*, not swallowed.


In [ ]:
MAX_ATTR = 2000  # truncation budget for bulky/sensitive payloads

def redact(text: str) -> str:
    """Toy redaction: strip an obvious secret pattern before it lands on a span."""
    import re
    return re.sub(r"(sk-[A-Za-z0-9]{6,})", "[REDACTED]", text)

def traced_model_call(messages, turn_idx, rendered_prompt, **params):
    with tracer.start_as_current_span("llm.call") as span:
        span.set_attribute("gen_ai.request.model", MODEL)
        # The single highest-value attribute: what the model ACTUALLY saw.
        span.set_attribute("gen_ai.prompt.rendered", redact(rendered_prompt)[:MAX_ATTR])
        span.set_attribute("gen_ai.request.temperature", params.get("temperature", 1.0))
        resp = call_model(messages, turn_idx, **params)
        span.set_attribute("gen_ai.response.model", resp.model)
        span.set_attribute("gen_ai.usage.input_tokens", resp.usage.input_tokens)
        span.set_attribute("gen_ai.usage.output_tokens", resp.usage.output_tokens)
        span.set_attribute("gen_ai.response.finish_reason", resp.stop_reason)
        return resp

def traced_tool_call(name, **args):
    with tracer.start_as_current_span(f"tool.call {name}") as span:
        span.set_attribute("tool.name", name)
        span.set_attribute("tool.args", redact(json.dumps(args))[:MAX_ATTR])
        try:
            out = TOOLS[name](**args)
            span.set_attribute("tool.result.size", len(str(out)))
            return out
        except Exception as exc:
            span.record_exception(exc)               # captured IN the tree
            span.set_status(trace.Status(StatusCode.ERROR))
            raise

print("instrumentation defined: agent.run / llm.call / tool.call")

### Run the agent under one root span

`run_agent` opens the `agent.run` root and drives a tiny loop: lookup the account (tool), try to send the report (**fails once, retried**), then a final model turn. The `with` block guarantees the root span closes — and stamps `agent.steps` / `agent.outcome` — even if a step raises.


In [ ]:
def run_agent(user_id, session_id, user_request):
    reset_spans()
    _send_report_attempts["n"] = 0
    steps = 0
    with tracer.start_as_current_span("agent.run") as root:
        root.set_attribute("agent.name", "support-agent")
        root.set_attribute("user.id", user_id)        # an id, NEVER PII
        root.set_attribute("session.id", session_id)
        try:
            # Step 1: model decides to look up the account.
            traced_model_call([], 0, f"SYSTEM: support agent\nUSER: {user_request}")
            steps += 1
            acct = traced_tool_call("lookup_account", account_id="acct_42")

            # Step 2: model decides to send the report; first send fails, retry succeeds.
            traced_model_call([], 1, f"Account {acct['account_id']}: {acct['status']}")
            steps += 1
            try:
                traced_tool_call("send_report", to="ops@example.com", body="summary")
            except ConnectionError:
                traced_tool_call("send_report", to="ops@example.com", body="summary")

            # Step 3: final answer.
            final = traced_model_call([], 2, "Compose the final answer for the user.")
            steps += 1
            root.set_attribute("agent.steps", steps)
            root.set_attribute("agent.outcome", "ok")
            return final.text
        except Exception:
            root.set_attribute("agent.steps", steps)
            root.set_attribute("agent.outcome", "error")
            raise

answer = run_agent("user_8821", "sess_0001", "Send ops a status report for my account.")
print("agent answer:", answer)
print("spans captured:", len(exporter.get_finished_spans()))

### 🔮 Predict before you read the tree

We are about to print the span tree with each span's duration. The run made **three model calls** and **three tool calls** (because `send_report` failed once and was retried).

*Predict:* which **kind** of span do you expect to dominate latency — `llm.call` or `tool.call`? And how many spans total will hang off the `agent.run` root?

Write your guess down, then run the next two cells.


In [ ]:
def build_tree(spans):
    """Reconstruct parent->children from captured spans by span id."""
    by_id = {s.context.span_id: s for s in spans}
    children = {s.context.span_id: [] for s in spans}
    roots = []
    for s in spans:
        parent = s.parent.span_id if s.parent else None
        if parent in children:
            children[parent].append(s)
        else:
            roots.append(s)
    # Order children by start time so the tree reads chronologically.
    for sid in children:
        children[sid].sort(key=lambda s: s.start_time)
    return roots, children, by_id

def ms(span):
    return (span.end_time - span.start_time) / 1e6  # ns -> ms

def print_tree(spans):
    roots, children, _ = build_tree(spans)
    def walk(s, depth):
        status = "" if str(s.status.status_code) == "StatusCode.UNSET" else f"  [{s.status.status_code}]"
        toks = ""
        a = s.attributes or {}
        if "gen_ai.usage.input_tokens" in a:
            toks = f"  in={a['gen_ai.usage.input_tokens']} out={a['gen_ai.usage.output_tokens']}"
        print("  " * depth + f"• {s.name}  ({ms(s):.1f} ms){toks}{status}")
        for c in children.get(s.context.span_id, []):
            walk(c, depth + 1)
    for r in sorted(roots, key=lambda s: s.start_time):
        walk(r, 0)

print_tree(exporter.get_finished_spans())

**What you just saw.** Read it like the chapter's `agent-trace` figure. The root `agent.run` carries the run-level attributes; six children hang off it (3 × `llm.call`, 3 × `tool.call`). Crucially, the **first `tool.call send_report` is marked ERROR and is followed by a successful retry** — exactly the event a final-answer-only log would have erased. (In `MOCK=1` the durations are sub-millisecond because nothing really sleeps; the *shape* of the tree is what matters. Under a live model, the `llm.call` spans dominate latency — network + generation — which is the usual answer to the prediction.)


### The failed span, up close

Because we called `record_exception` and set `StatusCode.ERROR`, the failure is *in the tree* as a recorded event with the exception type and message — not a line lost in stdout. This is what lets you debug a transient failure that never reproduces.


In [ ]:
for s in exporter.get_finished_spans():
    if str(s.status.status_code) == "StatusCode.ERROR":
        print("failing span:", s.name)
        for ev in s.events:
            etype = ev.attributes.get("exception.type")
            emsg = ev.attributes.get("exception.message")
            print(f"  event: {ev.name}  {etype}: {emsg}")

### Cost accounting from span attributes

Cost is a metered resource attached to every model call. We build the book's `span_cost` from `gen_ai.usage.*` plus an **illustrative** prices table (prices change — load from config in production), then sum per-run cost.

This is also where the chapter's scariest failure mode lives: the **cost incident** — a looping agent burns hundreds of dollars in an afternoon *with no error in the logs*. The defense is an in-code per-run token ceiling that stops the loop, not the monthly invoice.


In [ ]:
# (input $/MTok, output $/MTok) — illustrative; load from config, prices change.
PRICES = {"claude-sonnet-4-6": (3.00, 15.00)}

def span_cost(span) -> float:
    a = span.attributes
    p_in, p_out = PRICES[a["gen_ai.request.model"]]
    return (a["gen_ai.usage.input_tokens"] * p_in
            + a["gen_ai.usage.output_tokens"] * p_out) / 1e6

def run_cost(spans):
    return sum(span_cost(s) for s in spans
               if s.name == "llm.call" and "gen_ai.usage.input_tokens" in (s.attributes or {}))

def run_tokens(spans):
    return sum((s.attributes["gen_ai.usage.input_tokens"] + s.attributes["gen_ai.usage.output_tokens"])
               for s in spans if s.name == "llm.call")

spans = exporter.get_finished_spans()
print(f"tokens this run : {run_tokens(spans):,}")
print(f"cost this run   : ${run_cost(spans):.5f}")

# The cost-incident defense: a hard ceiling in code, checked per run.
TOKEN_CEILING = 50_000
if run_tokens(spans) > TOKEN_CEILING:
    print("⚠️  per-run token ceiling exceeded — a real loop would be halted here")
else:
    print(f"within ceiling ({TOKEN_CEILING:,} tokens) — loop allowed to continue")

### Context propagation across a sub-agent boundary

Two practices make traces vastly more useful; the first is *propagate context everywhere*. When a request fans out to a Celery worker (Ch 31) or a sub-agent, you must carry the trace context across the boundary so the distributed run stays **one tree**. Here we simulate the boundary: extract the current context, hand it to a "sub-agent" function (as a real worker would receive it over the wire), and reattach it so its spans nest under the same root.


In [ ]:
from opentelemetry import context as otel_context
from opentelemetry.propagate import extract, inject

def sub_agent(carrier):
    """Simulates a Celery worker: rehydrates context from the carrier, then traces."""
    ctx = extract(carrier)  # parent context arrives as a dict (e.g. message headers)
    token = otel_context.attach(ctx)
    try:
        with tracer.start_as_current_span("sub_agent.summarize") as span:
            span.set_attribute("agent.name", "summarizer")
            return "summary ok"
    finally:
        otel_context.detach(token)

reset_spans()
with tracer.start_as_current_span("agent.run") as root:
    root.set_attribute("agent.name", "support-agent")
    carrier = {}
    inject(carrier)                 # serialize current context into the carrier (headers)
    print("carrier handed across boundary:", carrier)
    sub_agent(carrier)              # runs "elsewhere" but joins the same trace

# Same trace id on parent and child => one tree across the boundary.
ids = {hex(s.context.trace_id) for s in exporter.get_finished_spans()}
print("distinct trace ids:", len(ids), "->", "ONE TREE" if len(ids) == 1 else "BROKEN")
print_tree(exporter.get_finished_spans())

### ⚠️ Pitfall: logging the prompt *template*, not the *rendered prompt*

The single most common — and most painful — gap. Three weeks after deploy a trace shows the agent inventing a refund policy, and you cannot tell whether retrieval served garbage or the template interpolated an empty string — **because all you stored was `support_v3.j2`**. Store what the model *actually saw*. Below, the template logs identically for two very different runs; the rendered prompt does not.


In [ ]:
template = "support_v3.j2"
good_context = "Refund window: 30 days from delivery."
bad_context = ""  # retrieval returned nothing; the template silently fills an empty string

def render(ctx):
    return f"[{template}] Use ONLY this policy:\n{ctx}\nAnswer the user."

print("What a template-only log shows for BOTH runs:")
print(" ", template)
print(" ", template)
print("\nWhat a rendered-prompt log shows (the difference that explains the bug):")
print("  GOOD ->", repr(render(good_context)))
print("  BAD  ->", repr(render(bad_context)))
print("\nThe BAD run handed the model an empty policy — invisible unless you store the rendered prompt.")

### ⚠️ Pitfall: payloads are bulky and sensitive

The day-one *capture wish list* — rendered prompt + version, model id + every sampling param, retrieved chunks with scores, full tool args + raw results, plan/reasoning summary, git SHA, feedback/eval verdict — is exactly the data that is expensive to reconstruct after an incident. But you cannot capture it naively: prompts and results are bulky and may carry PII or secrets. The answer is **capture with truncation, redaction, and a retention policy** — not "log nothing" and not "log everything forever." We already applied `redact()` and `[:MAX_ATTR]` to every payload attribute above; here's the redaction proving itself.


In [ ]:
leaky = 'tool call with api_key=sk-abcdef123456 and user note about order #42'
print("raw     :", leaky)
print("on-span :", redact(leaky)[:MAX_ATTR])
print("\nRetention is policy, not code: e.g. keep full payloads 14 days, then attributes-only.")

## 🎯 Senior lens

Instrument **once** with OpenTelemetry and treat the platform as a *swappable backend*. The expensive, durable asset is your instrumentation — the span names, the GenAI attributes, the rendered-prompt-and-cost habits — not the dashboard rendering it. We never touched a vendor SDK in this notebook; we changed one line (the exporter) and everything else is portable to Langfuse, LangSmith, Phoenix, or a raw OTLP collector. A team that couples its instrumentation to one platform's client pays that migration twice; a team that emits clean OTel pays it once, in a config file.


## Recap

- A **trace** is a flight recorder for one agent run — a tree of spans, one user request → one trace.
- Instrument three spans with the book's shapes: `agent.run` (run-level ids + outcome), `llm.call` (OTel **GenAI** attributes), `tool.call` (name, truncated args, errors recorded *in the tree*).
- The failed-then-retried `send_report` shows why the trace beats a final-answer log: it preserves what the run *actually did*.
- **Cost** is computed from `gen_ai.usage.*` via `span_cost`; defend against the silent **cost incident** with an in-code per-run token ceiling.
- **Propagate context** across sub-agent / worker boundaries so a distributed run stays one tree.
- Capture the rendered prompt (not the template) with **truncation, redaction, and retention** — the highest-value attribute, captured safely.


## Exercises

Each exercise *changes* something and asks you to **predict** the result before running.

1. **Add a retrieval span.** Insert a `tool.call retrieve` (or a dedicated `retrieval` span) before the first model call, with attributes `retrieval.chunk_count` and `retrieval.top_score`. *Predict* how many children now hang off the root, then verify with `print_tree`.
2. **Trip the cost ceiling.** Lower `TOKEN_CEILING` to `4000` and re-run the cost cell. *Predict* whether this run breaches it, then confirm — and explain why a real loop would halt here rather than at the invoice.
3. **Break, then fix, propagation.** In `sub_agent`, comment out the `extract(carrier)` / `attach` lines so the span starts with no parent. *Predict* how many distinct trace ids you'll see, run it, then restore the lines and confirm you're back to one tree.
4. **Make the model turn fail.** Add a fourth canned turn with `stop_reason="error"` and have `run_agent` set `agent.outcome="error"`. *Predict* what the root span's attributes look like, then read them off the captured spans.

In [ ]:
# Exercise 1: add a retrieval span, then print_tree again.


In [ ]:
# Exercise 2: lower TOKEN_CEILING and re-check run_tokens vs the ceiling.


In [ ]:
# Exercises 3 & 4: break/fix propagation; add an error turn and read root attributes.


## Next

- **Next notebook:** [`23-02-debugging-and-dashboards.ipynb`](./23-02-debugging-and-dashboards.ipynb) — debug a non-reproducible failure *from the trace alone*, then aggregate run- and **session**-level metrics and set page-vs-ticket alerting.
- **Blueprint (the production version of what you just built):** [`../../../blueprints/observability-stack/`](../../../blueprints/observability-stack/) — instrument once with OTel, swap the backend, with real cost accounting and dashboards.
- **Capstone:** this advances [`../../../capstone/`](../../../capstone/) `telemetry.py` — the instrumented agent loop whose `agent.run` / `llm.call` / `tool.call` spans gate and explain every run (checkpoint `ch23-observability`).
- **Book:** §23.1 (tracing with OTel), §23.3 (logging, metrics & cost accounting), §23.2 (platforms as a swappable backend).